# Banking77 pilot: train, evaluate, back up and release

Owner-specific release runner for `JatinMallarapu`. Use an existing **free T4 x2** session with Internet enabled and `HF_TOKEN` attached through Kaggle Secrets. Do not put a token in a cell.

The first release uses a balanced **200-source teacher budget**, all **8,992 gold training queries**, three seeds per student, and the untouched official test split for final comparisons. The larger all-source distillation study remains future work.

The runner verifies a private checkpoint upload/download before training. It publishes the two authorized public model repositories only after training, complete evaluations, an instruction-attack audit, and release evidence checks. Anonymous downloads and inference must then pass. No paid instance is created.

Use **Save Version → Save & Run All (Commit)**. The final cell stays busy while the workflow runs; do not detach subprocesses into an idle draft session. Environments and model caches stay under `/kaggle/temp`, outside saved notebook output.


In [ ]:
import os, pathlib, subprocess, sys
root = pathlib.Path('/kaggle/working/Banking77Distillation')
if not root.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jatinmallarapu/Banking77Distillation.git', str(root)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.13.9'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', '/kaggle/temp/b77-venv'], check=True)
gpu_python = '/kaggle/temp/b77-venv/bin/python'
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', gpu_python, '-e', str(root) + '[train,teacher,teacher-eval,dev]'], check=True)
gpu_env = dict(os.environ, PATH='/kaggle/temp/b77-venv/bin:' + os.environ['PATH'], HF_HOME='/kaggle/temp/huggingface')
gpu_env.pop('HF_TOKEN', None)
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
from kaggle_secrets import UserSecretsClient

# The secret is passed through stdin, never command arguments, logs or files.
# Keep this foreground call running or use Save & Run All; closing the browser
# does not stop a committed Kaggle job.
subprocess.run(
    [gpu_python, '-m', 'b77.pilot_pipeline', '--token-stdin', '--max-hours', '10.5'],
    cwd=root, env=gpu_env, text=True, check=True,
    input=UserSecretsClient().get_secret('HF_TOKEN'),
)
